In [ ]:
import os
import logging
import argparse
import numpy as np
import pandas as pd
from datetime import timedelta, datetime
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import sys
sys.argv = ['']
import polars as pl

In [ ]:
# tab_path = "/opt/data/commonfilesharePHI/ldiao/trash/ckd_tab_m_v1_full_scan_csv_f32_i32/ckd_processed_tab.csv"
tab_path = "/opt/data/commonfilesharePHI/ldiao/ckd_project/ckd_tab_full_f32_i16_read_icd_stage_filter_v4/ckd_processed_tab.csv"

In [ ]:
# metadata = pd.read_csv(tab_path, parse_dates=["EventDate"])
metadata = pl.read_csv(tab_path, schema_overrides={"CKD_stage": pl.Utf8})

In [ ]:
# polars


In [ ]:
metadata.head()

In [ ]:
from tableone import TableOne, load_dataset
import pandas as pd

In [ ]:
len(metadata['PatientID'].unique())

In [ ]:
metadata.shape

In [ ]:
def clean_ckd_stage(value):
    try:
        return int(value)
    except: # ValueError:
        if isinstance(value, str) and value[0].isdigit():
            return int(value[0])
        else:
            return np.nan
            
def filter_patients_by_ckd_stage(df, ckd_stage_col, patient_id_col='PatientID'):
    initial_patients = df[patient_id_col].nunique()
    # Filter for visits where CKD stage is 3 or higher
    df_at_or_above_stage_3 = df[df[ckd_stage_col] >= 3]
    # Get unique PatientIDs from this filtered DataFrame
    patient_ids_to_keep = set(df_at_or_above_stage_3[patient_id_col].unique())
    
    patients_removed = initial_patients - len(patient_ids_to_keep)

    return patient_ids_to_keep        

def find_CKD_stage_progression(df):
    df_sorted = df.sort_values(by=['PatientID', 'EventDate_dt'])
    
    # difference in CKD_stage for each patient
    df_sorted['stage_diff'] = df_sorted.groupby('PatientID')['CKD_stage_clean'].diff()

    # filter where the stage difference is positive (i.e., increased)
    df_increased = df_sorted[df_sorted['stage_diff'] > 0].copy()

    # retreive previous CKD_stage for context
    df_increased['previous_CKD_stage'] = df_sorted.groupby('PatientID')['CKD_stage_clean'].shift(1)
    
    # rename relevant columns
    result = df_increased[['PatientID', 'EventDate_dt', 'previous_CKD_stage', 'CKD_stage_clean']]
    result.rename(columns={'CKD_stage_clean': 'new_CKD_stage'}, inplace=True)
    
    return result

def unique_patient_ckd_counts(df):
    # Select only the necessary columns and drop duplicate rows based on PatientID
    # to ensure each patient is counted only once for their CKD stage.
    unique_patients_ckd = df[['PatientID', 'CKD_stage_clean']].drop_duplicates(subset=['PatientID'])

    # Count the occurrences of each CKD stage among these unique patients
    ckd_stage_counts = unique_patients_ckd['CKD_stage_clean'].value_counts()

    return ckd_stage_counts.sort_index()

In [ ]:
metadata['CKD_stage_clean'] = metadata['CKD_stage'].apply(clean_ckd_stage)
metadata = metadata.sort_values(by=['PatientID', 'EventDate'])
metadata['CKD_stage_clean'] = metadata.groupby('PatientID')['CKD_stage_clean'].bfill().ffill()
metadata = metadata.dropna(subset=['CKD_stage_clean'])
metadata['CKD_stage_clean'] = metadata['CKD_stage_clean'].astype(int)
metadata['label'] = metadata['CKD_stage_clean'].apply(lambda x: 1 if x >= 4 else 0)

In [ ]:
metadata.head()

In [ ]:
metadata['EventDate_dt'] = pd.to_datetime(metadata['EventDate'], errors='coerce')

In [ ]:
increased_stages_df = find_CKD_stage_progression(metadata)
increased_stages_df.shape

In [ ]:
increased_stages_df.head()

In [ ]:
# total number of patients
npatients = len(metadata['PatientID'].unique())
print(npatients)

In [ ]:
df = metadata
df.shape
# check nan rows
# nan_mean = df.isnull().mean()
# nan_mean

In [ ]:
df_patients = filter_patients_by_ckd_stage(metadata, 'CKD_stage_clean')
df = df[df["PatientID"].isin(df_patients)].copy()

In [ ]:
# total number of patients after filtering
npatients = len(df['PatientID'].unique())
print(npatients)

In [ ]:
df.shape

In [ ]:
increased_stages_df = find_CKD_stage_progression(df)
increased_stages_df.shape

In [ ]:
increased_stages_df['new_CKD_stage'].hist()

In [ ]:
# compare progression to true values (pre filtering)
df1 = pd.read_csv("./365day_future_prediction_outputs_50/LSTM_365DayFutureTarget_detailed_outputs.csv")
df1.head()

In [ ]:
len(df1['PatientID'].unique())

In [ ]:
df1['cl_true_label'].sum()

In [ ]:
# compare progression to true values (post filtering)
df2 = pd.read_csv("./365day_future_prediction_outputs_50_filter_stage_3/LSTM_365DayFutureTarget_detailed_outputs.csv")
df2.head()

In [ ]:
df2.shape

In [ ]:
len(df2['PatientID'].unique())

In [ ]:
df2['true_label'].sum()

In [ ]:
threshold = 0.0239
sum(df2['prob_positive'] > threshold) # use auc

In [ ]:
# pre filtering
patients_by_stage = unique_patient_ckd_counts(metadata)
patients_by_stage
ax = patients_by_stage[:20].plot(kind='bar', x='CKD_stage', grid=True)
ax.tick_params(axis='x', labelrotation=0)
ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax.set_title(' Patient Count by CKD Stage')
ax.set_xlabel("year")
ax.set_ylabel("patient count")

In [ ]:
# post filtering
patients_by_stage = unique_patient_ckd_counts(df)
patients_by_stage
ax = patients_by_stage[:20].plot(kind='bar', x='CKD_stage', grid=True)
ax.tick_params(axis='x', labelrotation=0)
ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax.set_title(' Patient Count by CKD Stage')
ax.set_xlabel("year")
ax.set_ylabel("patient count")